# Collocations and Keyword Extraction

The narrative companion to formalNLP's twenty-sixth topic. Every number here comes from
`collocations_keyword_extraction.py`, whose harness asserts each claim the page makes. The finance corpus is Topic 3's
training text, rebuilt from its own generator so that every token carries its provenance: a word drawn from a
Pitman–Yor restaurant (D), a word inside an entity name (E), or template text (L). That provenance is the answer key.
Run from the repository root.

In [ ]:
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path("notebooks/collocations-keyword-extraction").resolve()))
import collocations_keyword_extraction as ck

np.set_printoptions(precision=4, suppress=True)
b = ck.fin_bigrams()
print("bigram positions N =", int(b["n"]), "| bigram types:", len(b["types"]))
print({k: int((b["cls"] == k).sum()) for k in ("fixed", "null", "topical", "boundary")})

## 1. The 2×2 table as a point (Def 1, Def 2, Def 3, Thm 1, Prop 1)

A bigram $(w_1, w_2)$ splits the $N$ bigram positions into four cells: $a = c(w_1 w_2)$, $b = c(w_1\,\neg w_2)$,
$c' = c(\neg w_1\, w_2)$, $d$ the rest. Divided by $N$, the table is a point $\hat p$ of the tetrahedron $\Delta^3$. The
tables with independent rows and columns form the saddle $p_{11}p_{22} = p_{12}p_{21}$. The product of $\hat p$'s
margins is its KL projection onto that saddle (Thm 1), so

$$G^2 = 2N\,\mathrm{KL}(\hat p \,\|\, \hat p_r \otimes \hat p_c) = 2N\,\hat I(X;Y), \qquad X^2 = N\,\chi^2(\hat p \,\|\, \hat p_r \otimes \hat p_c) = N\hat\varphi^2.$$

In [ ]:
rng = np.random.default_rng(0)
p = rng.dirichlet(np.ones(4)).reshape(2, 2)
q = ck.independence_projection(p)
print("product of margins:\n", q)
print("numerical KL projection:\n", ck.projection_by_solver(p))
t = ck.Table(37, 120, 90, 1000)
s = ck.scores(t)
pt = t.cells() / t.n
print("G² =", float(s["g2"]), " 2N·KL =", 2 * 1000 * ck.kl(pt, ck.independence_projection(pt)))
print("X² =", float(s["x2"]), " N·φ² =", 1000 * float(ck.phi(t)) ** 2)

## 2. Rare pairs in closed form (Prop 2, Prop 3)

Two words seen once each, and only together: the table $[[1, 0], [0, N-1]]$. PMI reaches its maximum $\log N$, $X^2$
reaches its maximum $N$, while $G^2 = 2\log N + 2 - 1/N - O(N^{-2})$ and the exact $p$-value is $1/N$. Among perfect
pairs $[[k, 0], [0, N-k]]$, PMI $= \log(N/k)$ *falls* with $k$, $X^2 = N$ for every $k$, and $G^2 = 2N\,H(k/N)$ rises.

In [ ]:
print(ck.hapax_forms(ck.RARE_N))
pp = ck.perfect_pair(np.arange(1, 8), ck.RARE_N)
for key in ("pmi", "x2", "g2", "t", "neglog_p"):
    print(f"{key:9}", pp[key])

## 3. Tests and their sizes (Lemma 1, Def 4, Thm 2, Prop 4)

Given its margins, $a$ is hypergeometric under independence (Lemma 1), so Fisher's test is exact. $X^2$ and $G^2$ are
asymptotically $\chi^2_1$ (Thm 2), but their exact sizes at small expected counts $E_{11} = rc/N$ are far from 0.05: $X^2$
overshoots near $E \approx 0.2$, $G^2$ near $E \approx 2$, and the exact test never exceeds $\alpha$.

In [ ]:
for n, d in zip(ck.THM2_N, ck.thm2_draws()):
    print(n, "median |G² − X²| =", np.median(np.abs(d["g2"] - d["x2"])))
sc = ck._size_dense()
for k in ("x2", "g2", "fisher"):
    i = int(np.argmax(sc[k]))
    print(f"{k:7} max exact size {sc[k][i]:.4f} at E = {sc['expected'][i]:.4f}")

## 4. Testing only what you saw (Prop 5)

With one-sided $p(a) = P(A \ge a)$, the chance a pair is seen at all is $P(A \ge 1) = p(1)$. When that is at most
$\alpha$, *every* sighting rejects: $P(\text{reject} \mid \text{seen}) = 1$, while $P(\text{reject}) = P(\text{seen}) \le \alpha$. A
list of the bigrams one observed is exactly such a selection.

In [ ]:
dd = ck.dd_stratum()
for r in (1, 5, 10, 20, 40):
    print(r, {k: round(v, 4) for k, v in ck.selection(r, r, dd["n"], ck.ALPHA).items()})

In the corpus: inside the draw–draw stratum, two adjacent drawn words are independent given the sector, so a pair
with a shared-vocabulary word on either side is a true null. Benjamini–Hochberg over the observed pairs rejects most of
them; over all candidate pairs, it rejects none (and finds no topical pair either).

In [ ]:
fs = ck.fin_selection()
print("D–D positions", fs["n"], "| types", fs["types"], "| candidate pairs", fs["m_all"], "| seen once", round(fs["seen_once"], 4))
print("BH over observed pairs:", fs["observed"])
print("BH over all pairs:     ", fs["all"])
print("pooled table, BH over observed bigrams:", fs["pooled_observed"])

## 5. Ranking the corpus's bigrams

Precision@$k$ against the fixed phrases (template text and names), with ties broken at random in expectation. PMI's top
97 pairs tie at $\log N$ and are almost all null or topical pairs seen once; every perfect pair ties at $X^2 = N$.

In [ ]:
r = ck.fin_rankings(ck.RANK_KS)
for m in ck.MEASURES:
    print(f"{m:7} tie block {r[m]['top_block']:4}", "precision", {k: round(v, 3) for k, v in r[m]["precision"].items()},
          "null+topical@50", round(r[m]["null"][50] + r[m]["topical"][50], 3))
print(r["pmi"]["top"][:6])
print(r["g2"]["top"][:6])

## 6. Keyness (Def 5, Prop 6)

Keyness is the same 2×2 table with the corpus as the second variable. Each sector's documents against the rest, with
entity words removed, finds the planted sector nouns; raw frequency finds template words. Against the EDGAR excerpts, the
synthetic corpus's keywords are its templates.

In [ ]:
kw = ck.fin_keyness(ck.KEY_K)
for s, v in kw.items():
    print(f"{s:12} docs {v['docs']:3}  G² {v['g2']:.2f}  log-ratio {v['log_ratio']:.2f}  freq {v['freq']:.2f}  ", v["top"][:6])
ek = ck.fin_edgar_keyness()
print("synthetic vs EDGAR:", [w for w, _ in ek["top"][:10]], "template share of top 50:", ek["template_share"])

## 7. TextRank (Def 6, Def 7, Thm 3, Thm 4, Prop 7)

PageRank with damping $d$ has one stationary distribution, positive, and power iteration contracts in $\ell_1$ by $d$
per step. On the undirected window-2 graph a word's strength is $2c(w) - e(w) - 2s(w)$, and as $d \to 1$ PageRank tends to
each word's share of the strength: TextRank, with no reference corpus, tracks frequency.

In [ ]:
w = ck.random_graph(60, 0.05, 3911, directed=True)
pi, steps = ck.pagerank(w, 0.85, 1e-15)
print("power iteration vs dense eigenvector:", np.abs(pi - ck.pagerank_dense(w, 0.85)).max())
print("|λ₂| of the Google matrix:", np.sort(np.abs(np.linalg.eigvals(ck.google_matrix(w, 0.85))))[-2], "≤ d = 0.85")
print("strength identity mismatches:", ck.strength_identity(ck.ng.corpus()["train"], ck.textrank_stop()))
tr = ck.fin_textrank(ck.KEY_K, ck.TR_D)
for s, v in tr.items():
    x = v["weighted"][0.85]
    print(f"{s:12} TextRank P@25 {x['precision']:.2f}  Spearman with frequency {x['spearman']:.3f}  "
          f"frequency P@25 {v['freq']:.2f}  keyness P@25 {kw[s]['g2']:.2f}  iterations {x['iterations']}")

## 8. The harness

Every claim above is a test in the `.py`; running it prints the page's numbers at six decimals.

In [ ]:
for name, fn in [(n, f) for n, f in vars(ck).items() if n.startswith("test_") and callable(f)]:
    fn()
print("all tests passed")